# Mineração de Dados — Projeto Final
## Avaliação dos Problemas de SQL da Plataforma Beecrowd como Instrumento de Ensino: uma Abordagem por Clusterização e Regras de Associação

**Programa:** PPGCO — Faculdade de Computação / UFU · **Disciplina:** Mineração de Dados

Este notebook aplica o fluxo completo de mineração de dados a um dataset de **51 problemas de SQL**
da plataforma Beecrowd (ex-URI Online Judge), anotados com **21 conceitos de SQL** e atributos de
identificação, popularidade e complexidade. Além da mineração exploratória, o objetivo pedagógico é
apoiar a **escolha de exercícios por tópico**, a partir da relação entre problemas e conceitos
exigidos na solução.

### Questões de pesquisa
- **QP1** — A plataforma Beecrowd, no recorte analisado, é um instrumento adequado para o ensino de SQL?
- **QP2** — O nível de dificuldade declarado é coerente com a complexidade técnica real dos problemas?
- **QP3** — Os problemas cobrem suficientemente os conceitos de SQL (básico → avançado) para as necessidades curriculares?

### Técnicas de Aprendizado de Máquina aplicadas
1. **Clusterização (K-Means)** — descoberta não supervisionada de grupos de problemas por perfil conceitual.
2. **Regras de associação (Apriori)** — combinações de conceitos que coexistem no mesmo SELECT (sequências de ensino).


## 0. Ambiente (Google Colab)


In [ ]:
!pip -q install mlxtend

import os, warnings
warnings.filterwarnings("ignore")

CSV = "dataset_beecrowd_refinado-final.csv"

if os.path.exists(CSV):
    print("Dataset disponível: True")
else:
    print("Dataset disponível: False")
    print()
    print(f"O arquivo '{CSV}' ainda não está no ambiente.")
    print("Carregue-o pelo painel de Arquivos (ícone de pasta 📁 -> botão de upload ⬆)")
    print("e execute esta célula novamente.")

## 1. Configuração e carga dos dados


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import pearsonr, spearmanr
from mlxtend.frequent_patterns import apriori, association_rules

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

df = pd.read_csv("dataset_beecrowd_refinado-final.csv")
print("Dimensões:", df.shape)
df.head(3)

In [ ]:
usa_cols = [c for c in df.columns if c.startswith("usa_")]
print(f"{len(usa_cols)} conceitos anotados")

ROTULO = {
    "usa_where": "WHERE", "usa_order_by": "ORDER BY", "usa_distinct": "DISTINCT",
    "usa_like": "LIKE", "usa_in": "IN", "usa_between": "BETWEEN",
    "usa_agregacao": "Agregação", "usa_join": "JOIN", "usa_group_by": "GROUP BY",
    "usa_having": "HAVING", "usa_subconsulta": "Subconsulta",
    "usa_subconsulta_correlacionada": "Subcons. correl.", "usa_union": "UNION",
    "usa_case": "CASE", "usa_funcao_data": "Função de data",
    "usa_tratamento_null": "Tratamento de NULL", "usa_cast": "CAST",
    "usa_recursao": "Recursão", "usa_funcao_janela": "Função de janela",
    "usa_pivot": "PIVOT", "usa_funcao_string": "Função de string",
}
def rotular(cols):
    return [ROTULO.get(c, c) for c in cols]

## 2. Pré-processamento e verificação de qualidade


In [ ]:
info = pd.DataFrame({"dtype": df.dtypes.astype(str), "n_nulos": df.isna().sum()})
print("Colunas com valores ausentes:")
print(info[info.n_nulos > 0] if info.n_nulos.sum() else "  (nenhuma)")

soma_flags = df[usa_cols].sum(axis=1)
inconsistentes = int((df["num_conceitos_total"] != soma_flags).sum())
orfaos = int((soma_flags == 0).sum())
print(f"\nLinhas inconsistentes (total != soma flags): {inconsistentes}")
print(f"Problemas sem nenhum conceito (órfãos): {orfaos}")
assert inconsistentes == 0 and orfaos == 0, "Falha de integridade no dataset!"
print("Integridade OK.")

In [ ]:
num_cols = ["nivel", "num_resolvidos", "num_conceitos_total",
            "complexidade_tecnica", "discrepancia_nivel_complexidade"]
df[num_cols].describe().T.round(3)

## 3. Análise exploratória (EDA)


### 3.1 Cobertura conceitual — quantos problemas exercitam cada conceito? (QP3)


In [ ]:
freq = df[usa_cols].sum().sort_values(ascending=True)
freq.index = rotular(freq.index)

plt.figure(figsize=(9, 8))
cores = ["#C44E52" if v <= 2 else "#55A868" if v >= 10 else "#4C72B0" for v in freq.values]
plt.barh(freq.index, freq.values, color=cores)
for i, v in enumerate(freq.values):
    plt.text(v + 0.2, i, str(int(v)), va="center", fontsize=9)
plt.title("Cobertura conceitual: nº de problemas por conceito de SQL\n"
          "(vermelho ≤ 2 = cobertura rasa | verde ≥ 10 = bem coberto)")
plt.xlabel("nº de problemas"); plt.tight_layout(); plt.show()

### 3.2 Conceitos por faixa conceitual e por nível declarado (boxplots)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ordem = ["Básico", "Intermediário", "Avançado"]
dfp = df.copy()
dfp["faixa_conceitual"] = pd.Categorical(dfp["faixa_conceitual"], categories=ordem, ordered=True)
dfp = dfp.sort_values("faixa_conceitual")
sns.boxplot(data=dfp, x="faixa_conceitual", y="num_conceitos_total", ax=axes[0])
axes[0].set_title("Conceitos por faixa conceitual"); axes[0].set_xlabel("faixa conceitual")
sns.boxplot(data=dfp, x="nivel", y="num_conceitos_total", ax=axes[1])
axes[1].set_title("Conceitos por nível declarado (Beecrowd)"); axes[1].set_xlabel("nível")
plt.tight_layout(); plt.show()

### 3.3 Validação da classificação em faixas (Básico / Intermediário / Avançado)


In [ ]:
ordem = ["Básico", "Intermediário", "Avançado"]
AVANCADOS = ["usa_subconsulta", "usa_subconsulta_correlacionada",
             "usa_recursao", "usa_funcao_janela", "usa_pivot"]
AVANCADOS = [c for c in AVANCADOS if c in df.columns]
df["_n_avancados"] = df[AVANCADOS].sum(axis=1)

inconsist_basico = df[(df["faixa_conceitual"] == "Básico") & (df["_n_avancados"] > 0)]
inconsist_avanc  = df[(df["faixa_conceitual"] == "Avançado") & (df["_n_avancados"] == 0)]
print("Coerência faixa × conceitos complexos:")
print(f"  'Básico' que usam conceito avançado : {len(inconsist_basico)}  (esperado 0)")
print(f"  'Avançado' sem conceito complexo    : {len(inconsist_avanc)}  (esperado 0)")
print("  Média de conceitos avançados por faixa:")
print(df.groupby('faixa_conceitual')['_n_avancados'].mean().reindex(ordem).round(2).to_string())

cores = {"Básico": "#55A868", "Intermediário": "#4C72B0", "Avançado": "#C44E52"}
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
for f in ordem:
    d = df[df["faixa_conceitual"] == f]
    ax[0].scatter(d["num_conceitos_total"], d["_n_avancados"],
                  label=f, color=cores[f], s=80, edgecolor="k", alpha=0.75)
ax[0].set_xlabel("nº total de conceitos"); ax[0].set_ylabel("nº de conceitos avançados")
ax[0].set_title("Faixa × total de conceitos × conceitos avançados"); ax[0].legend(title="faixa")

dados = [df[df["faixa_conceitual"] == f]["nivel"] for f in ordem]
bp = ax[1].boxplot(dados, tick_labels=ordem, patch_artist=True)
for patch, f in zip(bp["boxes"], ordem):
    patch.set_facecolor(cores[f]); patch.set_alpha(0.75)
ax[1].set_ylabel("nível declarado (Beecrowd)"); ax[1].set_xlabel("faixa conceitual")
ax[1].set_title("Nível declarado por faixa conceitual")
plt.tight_layout(); plt.show()

df.drop(columns=["_n_avancados"], inplace=True)

## 4. Demonstração visual PROBLEMA × CONCEITO (núcleo pedagógico)


### 4.1 Matriz problema × conceito (heatmap)


In [ ]:
mat = df.sort_values("num_conceitos_total").set_index("titulo")[usa_cols]
mat.columns = rotular(mat.columns)

plt.figure(figsize=(13, 15))
sns.heatmap(mat, cmap=["#F2F2F2", "#2E5A87"], cbar=False, linewidths=0.4, linecolor="white")
plt.title("Matriz Problema × Conceito de SQL\n(célula escura = conceito exigido na solução)", fontsize=13)
plt.xlabel("Conceitos de SQL"); plt.ylabel("Problema (ordenado por nº de conceitos)")
plt.tight_layout(); plt.show()

### 4.2 Co-ocorrência de conceitos — quais conceitos aparecem juntos no mesmo SELECT?


In [ ]:
M = df[usa_cols].values
coocc = np.array(M.T @ M, dtype=int)
labels = rotular(usa_cols)
cooc_df = pd.DataFrame(coocc, index=labels, columns=labels)
for lab in labels:
    cooc_df.loc[lab, lab] = 0

plt.figure(figsize=(13, 11))
sns.heatmap(cooc_df, cmap="YlOrRd", annot=True, fmt="d", annot_kws={"size": 7},
            linewidths=0.3, cbar_kws={"label": "nº de problemas em que coexistem"})
plt.title("Co-ocorrência de conceitos no mesmo problema (SELECT)")
plt.tight_layout(); plt.show()

### 4.3 Densidade conceitual — problemas-síntese (Top 15)


In [ ]:
top = df.nlargest(15, "num_conceitos_total")[
    ["problema_id", "titulo", "nivel", "num_conceitos_total", "conceito_primario", "conceitos_lista"]
].reset_index(drop=True)
display(top)

plt.figure(figsize=(11, 7))
d = df.sort_values("num_conceitos_total", ascending=True).tail(15)
plt.barh(d["titulo"], d["num_conceitos_total"], color="#8172B3")
for i, (v, cp) in enumerate(zip(d["num_conceitos_total"], d["conceito_primario"])):
    plt.text(v + 0.1, i, f"{int(v)}  ·  {cp}", va="center", fontsize=8)
plt.title("Top 15 problemas por nº de conceitos combinados no mesmo SELECT\n(rótulo = total · conceito primário)")
plt.xlabel("nº de conceitos"); plt.tight_layout(); plt.show()

### 4.4 Mapa de escolha: conceito primário × faixa de complexidade


In [ ]:
pivot = pd.crosstab(df["conceito_primario"], df["faixa_conceitual"])
pivot = pivot.reindex(columns=["Básico", "Intermediário", "Avançado"], fill_value=0)
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(8, 9))
sns.heatmap(pivot, annot=True, fmt="d", cmap="Blues", linewidths=0.5,
            cbar_kws={"label": "nº de problemas"})
plt.title("Mapa de escolha: conceito primário × faixa de complexidade")
plt.ylabel("Conceito primário (tópico mais avançado do problema)"); plt.xlabel("Faixa")
plt.tight_layout(); plt.show()
pivot

## 5. Aprendizado de Máquina I — Clusterização (K-Means)


In [ ]:
features = usa_cols + ["nivel"]
X = df[features].fillna(0).astype(float)
X_scaled = StandardScaler().fit_transform(X)
print("Matriz de features:", X_scaled.shape)

### 5.1 Análise de hiperparâmetros — escolha de *k*


In [ ]:
ks = range(2, 8)
inercias, silhuetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_scaled)
    inercias.append(km.inertia_)
    silhuetas.append(silhouette_score(X_scaled, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(list(ks), inercias, "o-", color="#4C72B0"); ax[0].set_title("Método do cotovelo (inércia)")
ax[0].set_xlabel("k"); ax[0].set_ylabel("inércia")
ax[1].plot(list(ks), silhuetas, "o-", color="#C44E52"); ax[1].set_title("Coeficiente de silhueta")
ax[1].set_xlabel("k"); ax[1].set_ylabel("silhueta")
plt.tight_layout(); plt.show()

for k, s in zip(ks, silhuetas):
    print(f"k={k}: silhueta={s:.3f}")
melhor_k = list(ks)[int(np.argmax(silhuetas))]
print(f"\nMelhor k pela silhueta: {melhor_k}")

### 5.2 Ajuste final e perfil dos clusters


In [ ]:
k = 3
km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_scaled)

tmp = df.copy(); tmp["_lab"] = km.labels_
ordem_l = tmp.groupby("_lab")["num_conceitos_total"].mean().sort_values().index
remap = {c: i for i, c in enumerate(ordem_l)}
df["cluster_final"] = [remap[l] for l in km.labels_]

perfil = df.groupby("cluster_final").agg(
    n=("problema_id", "count"),
    nivel_medio=("nivel", "mean"),
    conceitos_medio=("num_conceitos_total", "mean"),
    resolvidos_medio=("num_resolvidos", "mean"),
    complexidade_media=("complexidade_tecnica", "mean"),
).round(2)
perfil

### 5.3 Visualização dos clusters via PCA (2D)


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
var = pca.explained_variance_ratio_

plt.figure(figsize=(9, 7))
sc = plt.scatter(coords[:, 0], coords[:, 1], c=df["cluster_final"],
                 cmap="Set1", s=90, edgecolor="k", alpha=0.85)
for i, r in df.iterrows():
    if r["num_conceitos_total"] >= 8:
        plt.annotate(str(r["titulo"])[:18], (coords[i, 0], coords[i, 1]),
                     fontsize=7, alpha=0.7, xytext=(3, 3), textcoords="offset points")
plt.title(f"Clusters de problemas (PCA 2D)\n"
          f"variância explicada: PC1={var[0]:.0%}, PC2={var[1]:.0%}")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.colorbar(sc, label="cluster (0=simples → 2=complexo)")
plt.tight_layout(); plt.show()

## 6. Aprendizado de Máquina II — Regras de Associação (Apriori)


### 6.1 Itemsets frequentes e geração das regras


In [ ]:
cesta = df[usa_cols].astype(bool)
cesta.columns = rotular(usa_cols)

freq_is = apriori(cesta, min_support=0.12, use_colnames=True, max_len=3)
print(f"Itemsets frequentes (sup≥0.12, até 3 itens): {len(freq_is)}")

regras = association_rules(freq_is, metric="confidence", min_threshold=0.6)
regras = regras[regras["lift"] > 1.2].copy()
regras["n_ant"] = regras["antecedents"].apply(len)
regras["n_con"] = regras["consequents"].apply(len)
regras = regras[(regras["n_ant"] <= 2) & (regras["n_con"] == 1)]
regras = regras.sort_values(["lift", "confidence"], ascending=False).reset_index(drop=True)

def _fmt(itemset): return " + ".join(sorted(itemset))
regras["Antecedente"] = regras["antecedents"].apply(_fmt)
regras["Consequente"] = regras["consequents"].apply(_fmt)

tabela = regras[["Antecedente", "Consequente", "support", "confidence", "lift"]].round(3)
tabela.columns = ["Antecedente", "Consequente", "Suporte", "Confiança", "Lift"]
print(f"Regras robustas e interpretáveis: {len(tabela)}")
display(tabela.head(15))

### 6.2 Visualização — suporte × confiança (cor = lift) e Top 12 por lift


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
scv = ax[0].scatter(regras["support"], regras["confidence"], c=regras["lift"],
                    cmap="viridis", s=90, edgecolor="k", alpha=0.85)
plt.colorbar(scv, ax=ax[0], label="lift")
ax[0].set_xlabel("suporte"); ax[0].set_ylabel("confiança")
ax[0].set_title("Regras de associação (cada ponto = uma regra; cor = lift)")

top_r = regras.head(12).iloc[::-1]
rot = top_r["Antecedente"] + "  →  " + top_r["Consequente"]
ax[1].barh(rot, top_r["lift"], color="#55A868", edgecolor="white")
for i, (l, c) in enumerate(zip(top_r["lift"], top_r["confidence"])):
    ax[1].text(l + 0.05, i, f"lift={l:.2f} · conf={c:.2f}", va="center", fontsize=8)
ax[1].set_xlabel("lift"); ax[1].set_title("Top 12 regras de associação (por lift)")
plt.tight_layout(); plt.show()

## 7. Respostas às questões de pesquisa


### QP2 — Coerência entre nível declarado e complexidade técnica real


In [ ]:
rho, p_s = spearmanr(df["nivel"], df["complexidade_tecnica"])
r, p_p   = pearsonr(df["nivel"], df["complexidade_tecnica"])
print("Correlação nível × complexidade técnica:")
print(f"  Pearson  r = {r:.3f} (p={p_p:.4f})")
print(f"  Spearman ρ = {rho:.3f} (p={p_s:.4f})")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.regplot(data=df, x="nivel", y="complexidade_tecnica", ax=ax[0],
            scatter_kws={"s": 60, "alpha": .6}, line_kws={"color": "red"})
ax[0].set_title(f"Nível declarado × complexidade real (ρ={rho:.2f})")

disc = df.reindex(df["discrepancia_nivel_complexidade"].abs().sort_values(ascending=False).index)
top_disc = disc.head(10)
ax[1].barh(top_disc["titulo"], top_disc["discrepancia_nivel_complexidade"],
           color=["#C44E52" if v > 0 else "#4C72B0" for v in top_disc["discrepancia_nivel_complexidade"]])
ax[1].axvline(0, color="k", lw=.8)
ax[1].set_title("Maiores discrepâncias\n(vermelho: nível superestima | azul: subestima)")
plt.tight_layout(); plt.show()

### QP1 & QP3 — Adequação e cobertura curricular


In [ ]:
freq_abs = df[usa_cols].sum()
rasos = freq_abs[freq_abs <= 2]
bem = freq_abs[freq_abs >= 10]
print(f"Total de conceitos anotados: {len(usa_cols)}")
print(f"Conceitos com cobertura RASA (≤2 problemas): {len(rasos)}")
print("  ->", rotular(rasos.index.tolist()))
print(f"Conceitos BEM cobertos (≥10 problemas): {len(bem)}")
print("  ->", rotular(bem.index.tolist()))
print("\nDistribuição por faixa conceitual:")
print(df["faixa_conceitual"].value_counts().reindex(["Básico", "Intermediário", "Avançado"]))